# LBM_CODE on a GPU — first port

**Nothing in this repository has ever been run on a GPU.** Every number in
`doc/lbm_code.tex` and in the README is the Kokkos *Threads* backend. This
notebook is a port attempt, not a benchmark: it exists to answer *does it
build, and does it still get the right answer* on a CUDA device.

The code is written for a GPU — Esoteric Pull exists to halve GPU memory
traffic, every kernel is a Kokkos lambda capturing by value with no `this` on
the device, and the diagnostics mirror device→host rather than reading device
views from host code — so there is reason to expect it to work. That is not the
same as knowing it does.

**Order matters.** Run `ctest` on the device *before* the 3D Taylor–Green case.
If `poiseuille`, `decaying_flows` and `galilean` pass, the port is essentially
sound; debugging a 3D turbulence case first is miserable.

Runtime on a free tier: ~25–40 min, nearly all of it compiling.


## 1. What GPU did we get?

Colab's free tier hands out whatever is spare, so the architecture has to be
detected rather than assumed. The compute capability maps to a `Kokkos_ARCH_*`
macro, and a wrong one usually shows up as a *runtime* launch failure rather
than a build error — which is a horrible way to lose an afternoon.


In [ ]:
import subprocess, re, os

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or
      'no nvidia-smi -- is the runtime set to GPU? Runtime > Change runtime type')
print(subprocess.run(['nvcc','--version'], capture_output=True, text=True).stdout)

cc = subprocess.run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],
                    capture_output=True, text=True).stdout.strip().split('\n')[0]

# compute capability -> Kokkos arch macro (without the Kokkos_ARCH_ prefix)
ARCH = {'7.0':'VOLTA70', '7.5':'TURING75', '8.0':'AMPERE80', '8.6':'AMPERE86',
        '8.9':'ADA89', '9.0':'HOPPER90'}
KOKKOS_ARCH = ARCH.get(cc)
print(f'compute capability {cc} -> Kokkos_ARCH_{KOKKOS_ARCH}')
assert KOKKOS_ARCH, f'unmapped compute capability {cc}; add it to ARCH above'

# CUDA 12+ is needed for C++20 through nvcc, which this code requires.
v = subprocess.run(['nvcc','--version'], capture_output=True, text=True).stdout
major = int(re.search(r'release (\d+)\.', v).group(1))
assert major >= 12, f'CUDA {major} cannot compile C++20; this code needs CUDA 12+'
print(f'CUDA {major}: OK for C++20')


## 2. Get the code

Pick whichever applies. If the repository is not on a remote yet, the tarball
route is the quick one: on your machine, `tar czf lbm.tgz LBM_CODE` and upload.


In [ ]:
MODE = 'clone'    # 'clone' | 'upload' | 'drive'
REPO_URL = 'https://github.com/alexderosis/LBM_CODE.git'

import shutil, glob
SRC = '/content/LBM_CODE'
shutil.rmtree(SRC, ignore_errors=True)

if MODE == 'clone':
    assert REPO_URL, 'set REPO_URL'
    !git clone --depth 1 $REPO_URL $SRC
elif MODE == 'upload':
    from google.colab import files
    up = files.upload()                      # choose your .tgz / .zip
    name = list(up)[0]
    os.makedirs('/content/_x', exist_ok=True)
    !tar xf "$name" -C /content/_x 2>/dev/null || unzip -q "$name" -d /content/_x
    root = [p for p in glob.glob('/content/_x/*') if os.path.isdir(p)][0]
    shutil.move(root, SRC)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copytree('/content/drive/MyDrive/LBM_CODE', SRC)

assert os.path.exists(f'{SRC}/CMakeLists.txt'), 'CMakeLists.txt not found at ' + SRC
print('code at', SRC)


## 3. Kokkos, built once and cached

Kokkos is installed separately rather than fetched by the project's
`FetchContent` path, because of a chicken-and-egg: Kokkos wants
`CMAKE_CXX_COMPILER` pointed at its own `nvcc_wrapper`, which does not exist
until Kokkos has been fetched. The project's CMake already prefers an installed
Kokkos via `find_package`, so this needs no changes to the code.

Set `CACHE_TO_DRIVE = True` to keep the install across sessions. A free-tier
disconnect otherwise costs the entire build again, which is the single most
annoying way to lose progress here.


In [ ]:
CACHE_TO_DRIVE = False
KOKKOS_VER = '5.2.1'   # must match LBM_KOKKOS_VERSION in the project CMakeLists

if CACHE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PREFIX = f'/content/drive/MyDrive/kokkos-{KOKKOS_VER}-cuda-{KOKKOS_ARCH}'
else:
    PREFIX = f'/content/kokkos-install'
os.environ['PREFIX'] = PREFIX
os.environ['KOKKOS_ARCH'] = KOKKOS_ARCH
os.environ['KOKKOS_VER'] = KOKKOS_VER
print('install prefix:', PREFIX)


In [ ]:
%%bash
set -e
if [ -x "$PREFIX/bin/nvcc_wrapper" ]; then
  echo "cached Kokkos found at $PREFIX -- skipping build"
  exit 0
fi
cd /content
rm -rf kokkos kb
git clone --depth 1 -b "$KOKKOS_VER" https://github.com/kokkos/kokkos.git
cmake -S kokkos -B kb -DCMAKE_BUILD_TYPE=Release \
      -DCMAKE_INSTALL_PREFIX="$PREFIX" \
      -DCMAKE_CXX_COMPILER=/content/kokkos/bin/nvcc_wrapper \
      -DCMAKE_CXX_STANDARD=20 \
      -DKokkos_ENABLE_SERIAL=ON \
      -DKokkos_ENABLE_CUDA=ON -DKokkos_ENABLE_CUDA_LAMBDA=ON \
      -DKokkos_ARCH_${KOKKOS_ARCH}=ON
cmake --build kb -j$(nproc) --target install
echo 'Kokkos installed'


## 4. Build the cases `ctest` needs --- and only those

**FP32, deliberately.** Consumer NVIDIA parts run FP64 at 1/32 to 1/64 of their
FP32 rate, so a default `double` build on one of these is routinely *slower than
a laptop CPU*. The FP32 caveat in the Known Limitations then applies: it cannot
resolve the finest convergence tests. That is the right trade here --- this is a
port check, not a convergence study.

**`lbm_app` is deliberately NOT built here.** It is the most expensive
translation unit in the tree by a wide margin: `src/app/main.cpp` instantiates
seven complete solver configurations --- two-lattice and Esoteric Pull, raw and
shifted storage, BGK, TRT, raw MRT and central moments --- every one dragging
the whole `MomentCollision` -> `ProductBasis` -> `run_step` template chain
through nvcc. Building it first, as an earlier version of this notebook did,
puts the *benchmark* ahead of the *correctness check* and delays the only result
that decides whether the port is sound. It is built in §9, where it is needed.

The build directory is configured once and reused, so the second build adds only
`lbm_app` rather than starting over.


In [ ]:
%%bash
set -e
cd /content/LBM_CODE
# Configured once and NOT wiped, so §9 can add lbm_app incrementally.
cmake -S . -B build-gpu -DCMAKE_BUILD_TYPE=Release \
      -DCMAKE_CXX_COMPILER="$PREFIX/bin/nvcc_wrapper" \
      -DCMAKE_PREFIX_PATH="$PREFIX" \
      -DLBM_PRECISION=float
cmake --build build-gpu -j$(nproc) \
      --target poiseuille decaying_flows galilean thermal mhd regularized \
               hartmann scalar_walls tgv3d
echo; echo 'built:'; ls build-gpu/validation | head -20


## 5. `ctest` on the device — do this before anything else

These are the cases with something to be right against. If they pass here, the
port is sound and TGV3D is worth running. If they fail, fix that first: a
3D turbulence case will not tell you *why* anything is wrong.


In [ ]:
%%bash
cd /content/LBM_CODE/build-gpu
ctest --output-on-failure 2>&1 | tail -25


## 6. CPU reference

The comparison that makes this meaningful. Same source, same case, same
precision — only the backend differs, so any disagreement is the port and
nothing else.

**Keep the shared case small.** Step count is `tmax·D/u0`, so the defaults in
`tgv3d` (`-d 64 -tmax 10`) mean 32,000 steps over 262k nodes — 8.4e9 node
updates, which is 6–15 minutes on two vCPUs and a poor bet on a free session
that has already spent half an hour compiling. 48³ to t\*=5 is 12,000 steps and
answers the same question in a couple of minutes. Stretch the GPU separately,
in §7, where it costs nothing.

**Compare to tolerance, not bitwise.** CPU results in this codebase are bitwise
reproducible across thread counts (verified: 1, 2 and 4 threads give an
identical md5), which makes them an exact reference — but a GPU reduction sums
in a different order and will differ in the last digits. Disagreement at 1e-6 is
expected; disagreement at 1e-2 is a bug.


In [ ]:
# Shared by the CPU reference and the GPU comparison run -- they MUST match.
D_CMP, TMAX_CMP, RE = 48, 5, 1600
print(f'comparison case: {D_CMP}^3, t*<={TMAX_CMP}, Re={RE} '
      f'-> {int(TMAX_CMP*D_CMP/0.02)} steps, {D_CMP**3} nodes')

# GPU-only stretch run in section 7. FP32 D3Q27 is ~108 B/node:
#   128^3 = 226 MB, 192^3 = 764 MB, 256^3 = 1.8 GB, before macroscopic fields.
D_BIG, TMAX_BIG = 128, 10
print(f'gpu stretch:     {D_BIG}^3, t*<={TMAX_BIG} '
      f'-> {int(TMAX_BIG*D_BIG/0.02)} steps, ~{D_BIG**3*108/1e6:.0f} MB')

import os
for k, v in dict(D_CMP=D_CMP, TMAX_CMP=TMAX_CMP, RE=RE,
                 D_BIG=D_BIG, TMAX_BIG=TMAX_BIG).items():
    os.environ[k] = str(v)


In [ ]:
%%bash
set -e
cd /content/LBM_CODE
# Serial build through the project's own FetchContent path -- no nvcc, so much
# quicker than the GPU build above, but it does compile Kokkos a second time.
cmake -S . -B build-cpu -DCMAKE_BUILD_TYPE=Release -DLBM_PRECISION=float \
      -DKokkos_ENABLE_SERIAL=ON > /dev/null
cmake --build build-cpu -j$(nproc) --target tgv3d lbm_app > /dev/null
mkdir -p results/E_tgv3d          # open_out() returns NULL if this is missing
rm -f results/E_tgv3d/*.dat
time ./build-cpu/validation/tgv3d -d $D_CMP -re $RE -tmax $TMAX_CMP
cp results/E_tgv3d/*.dat /content/cpu_ref.dat
echo; echo 'CPU reference saved'


## 7. 3D Taylor–Green on the GPU

First at the comparison settings, so §8 has something to check against; then a
larger run to see the device do something a laptop would not enjoy. There is no
MPI, so the big one has to fit in a single device's memory.


In [ ]:
%%bash
set -e
cd /content/LBM_CODE
mkdir -p results/E_tgv3d

echo "=== comparison case: ${D_CMP}^3 ==="
rm -f results/E_tgv3d/*.dat
time ./build-gpu/validation/tgv3d -d $D_CMP -re $RE -tmax $TMAX_CMP
cp results/E_tgv3d/*.dat /content/gpu.dat

echo; echo "=== stretch: ${D_BIG}^3 (GPU only, not compared) ==="
rm -f results/E_tgv3d/*.dat
time ./build-gpu/validation/tgv3d -d $D_BIG -re $RE -tmax $TMAX_BIG


## 8. Did the GPU get the same answer?


In [ ]:
import numpy as np, matplotlib.pyplot as plt

cpu = np.loadtxt('/content/cpu_ref.dat')
gpu = np.loadtxt('/content/gpu.dat')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(cpu[:,0], cpu[:,1], 'o-', label='CPU', mfc='none')
ax[0].plot(gpu[:,0], gpu[:,1], '.--', label='GPU')
ax[0].set_xlabel('t*'); ax[0].set_ylabel('E / E0')
ax[0].set_title('kinetic energy decay'); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(cpu[:,0], cpu[:,2], 'o-', label='CPU', mfc='none')
ax[1].plot(gpu[:,0], gpu[:,2], '.--', label='GPU')
ax[1].set_xlabel('t*'); ax[1].set_ylabel('enstrophy / enstrophy0')
ax[1].set_title('enstrophy'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

n = min(len(cpu), len(gpu))
dE = np.abs(gpu[:n,1] - cpu[:n,1]).max()
dP = np.abs(gpu[:n,2] - cpu[:n,2]).max()
print(f'max |dE/E0|        = {dE:.3e}')
print(f'max |dPsi/Psi0|    = {dP:.3e}')
print()
if max(dE, dP) < 1e-4:
    print('AGREES. Differences at this level are reduction order and FP32 round-off.')
elif max(dE, dP) < 1e-2:
    print('CLOSE but larger than round-off should give. Worth understanding before')
    print('trusting production runs -- try FP64 on both sides to see if it shrinks.')
else:
    print('DISAGREES. This is a port bug, not arithmetic. Go back to ctest and find')
    print('the smallest case that fails.')


## 9. MLUPS

`lbm_app` is the throughput benchmark: a fully periodic box, seven
streaming/storage/operator combinations, reporting million lattice updates per
second and the implied memory bandwidth. It fences either side of the timed
loop, so the numbers mean something on a device where kernel launches are
asynchronous — without that you measure the launch queue, not the work.

Three things to keep in mind reading the result:

* **Size the problem up.** The 96³ default is 884k nodes, which does not fill a
  modern GPU — you will be measuring launch overhead and occupancy limits as
  much as throughput. Sweep it.
* **This is a first port with no GPU tuning.** Esoteric Pull is a good starting
  point precisely because it halves memory traffic, but nothing here has ever
  been profiled on a device. Treat these as a baseline to improve on.
* **Watch the mass-drift column.** `bench()` fails the run if drift exceeds
  1e-4 in FP32. A fast wrong answer is not a result.


In [ ]:
%%bash
set -e
cd /content/LBM_CODE
# The expensive one, built last and on purpose -- see the note in §4.
cmake --build build-gpu -j$(nproc) --target lbm_app
echo 'lbm_app built'


In [ ]:
%%bash
set -e
cd /content/LBM_CODE
for N in 96 128 160; do
  echo "================ GPU, ${N}^3 ================"
  ./build-gpu/lbm_app -n $N -steps 100
done


In [ ]:
%%bash
cd /content/LBM_CODE
echo "================ CPU, 96^3 (same build, Serial backend) ================"
./build-cpu/lbm_app -n 96 -steps 20


The CPU line runs fewer steps on purpose — it is there for the ratio, not as a
CPU benchmark in its own right. Colab gives you two vCPUs, so do not read it as
representative of a real workstation: the figures in `doc/lbm_code.tex` were
measured on an Apple M1 with four threads and are the honest CPU reference.


## Notes

* **No MPI**, so a single device only.
* **No Metal backend in Kokkos**, so an Apple GPU is not a target for this.
* Free-tier GPU availability, session length and preemption policy change often.
  Kaggle Notebooks gives longer sessions and a more predictable weekly quota; a
  university HPC facility beats both for anything you intend to iterate on, and
  spares you rebuilding Kokkos every session.
* If timings are the goal rather than correctness, remember this is a first port
  with no GPU-specific tuning. Esoteric Pull is a good starting point precisely
  because it halves memory traffic, but nothing here has been profiled on a
  device.
